# Build Your Own Medical ASR Model

This tutorial walks you through building a personalized speech recognition model for German medical dictation. By the end, you'll have a finetuned ASR model that understands dermatology terminology.

**What you'll do:**
1. Generate synthetic medical text using a local LLM
2. Synthesize audio from that text (optionally with your own voice)
3. Package the data into training splits
4. Finetune Qwen3-ASR with LoRA on Apple Silicon
5. Evaluate: measure how much better your model got
6. Try it yourself: transcribe your own recordings

**Quick mode:** Everything below defaults to 50 samples and 20 training steps — enough to see the full pipeline in minutes. When you're ready for a real model, bump the numbers.

**Tested on:** Mac Studio M4 Max, 64 GB RAM.

**Prerequisites:**
- macOS with Apple Silicon (M1/M2/M3/M4)
- [Ollama](https://ollama.com) installed and running (`ollama serve`)
- Ollama model pulled: `ollama pull qwen3.5:35b`

## Stage 1: Generate Medical Text

We use a local LLM (via Ollama) to generate realistic German dermatology text — the kind of sentences a doctor would dictate into a recorder or a physician would say during a patient interaction. The built-in taxonomy covers 8 subspecialties with 54 conditions.

Each generated sample is automatically validated before being accepted:
- **Abbreviations** — rejected if it contains abbreviations (e.g. "mg" instead of "Milligramm"), since TTS would mispronounce them
- **Word count** — rejected if too short or too long for the requested length class
- **Formatting** — rejected if it contains markdown or other formatting
- **Duplicates** — rejected if too similar to an existing sample

Generation is **resumable**: if you interrupt and re-run, it picks up where it left off (data is appended to the same file). To start fresh, delete the `data/dataset.jsonl` file.

**Adapting to another specialty:** See the README for what to change in `asr/text.py`.

In [ ]:
import json
from pathlib import Path
from asr import generate_text

language = "de"  # language code for transcription and evaluation

# Load taxonomy — replace with your own for a different specialty
with open("asr/taxonomy.json") as f:
    taxonomy = json.load(f)

# Dictation contexts — what types of speech to generate
contexts = ["befund_diktat", "arzt_patient_dialog"]

# Domain-specific vocabulary seeded into prompts for lexical diversity
vocabulary = [
    "Erosion", "Induration", "Lichenifikation", "Mazerierung",
    "Teleangiektasien", "Komedonen", "Nikolski-Zeichen", "Poikilodermie",
    "Papel", "Pustel", "Vesikel", "Bulla", "Kruste", "Ulzeration",
    "Atrophie", "Exkoriation", "Hyperpigmentierung", "Depigmentierung",
    "Schuppung", "Rhagade", "Fistel", "Narbe", "Plaque",
]

# What content to include at each length class — adapt to your specialty
length_class_instructions = {
    "kurz": "Beschreibe nur Morphologie und Lokalisation.",
    "mittel": "Beschreibe Morphologie, Lokalisation und Therapieempfehlung.",
    "lang": "Beschreibe Befund, Differentialdiagnosen und diagnostische Abklaerung.",
    "sehr_lang": "Beschreibe Befund, Differentialdiagnosen, Therapie mit Medikamentennamen und Nachsorge.",
}

# Generate 50 samples for a quick test
text_dataset = generate_text(
    taxonomy=taxonomy,
    vocabulary=vocabulary,
    contexts=contexts,
    length_class_instructions=length_class_instructions,
    n_samples=50,
    # specialty="Dermatologie",              # change for a different specialty
    # ollama_host="http://localhost:11434",   # change if Ollama runs elsewhere
    # model="qwen3.5:35b",                   # change if using a different model
)
print(f"Text dataset: {text_dataset}")

## Stage 2: Synthesize Audio

Each text sample is converted to speech using [`Qwen3-TTS-12Hz-0.6B`](https://huggingface.co/mlx-community/Qwen3-TTS-12Hz-0.6B-Base-bf16) via mlx-audio. An augmented copy (with noise, speed, and pitch variation) is also generated to help the model generalize.

**Voice cloning (optional):** To synthesize audio in your own voice, record a 15-20 second WAV clip of yourself reading a medical text aloud. Pass it as `ref_audio` together with `ref_text` — the exact transcript of what you recorded. The model uses this pair to clone your voice characteristics.

**Note:** You will see warnings about `qwen3_tts` model type and an incorrect tokenizer regex pattern during model loading. These are harmless — they come from upstream libraries and do not affect audio generation.

In [ ]:
from asr import generate_audio

# Option A: Use the model's built-in voice (no recording needed)
audio_manifest = generate_audio(text_dataset)

# Option B: Clone your own voice (uncomment and adjust)
# audio_manifest = generate_audio(
#     text_dataset,
#     ref_audio=Path("my_voice.wav"),
#     ref_text="The exact transcript of what you said in my_voice.wav.",
# )

# Option C: Custom augmentation (uncomment and adjust)
# from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
# custom_augmentation = Compose([
#     AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.005, p=0.5),
#     TimeStretch(min_rate=0.85, max_rate=1.15, p=0.5),
#     PitchShift(min_semitones=-2, max_semitones=2, p=0.3),
# ])
# audio_manifest = generate_audio(text_dataset, augmentation=custom_augmentation)

print(f"Audio manifest: {audio_manifest}")

## Stage 3: Package Dataset

Splits the data into train/val/test sets, stratified by the categories defined in your taxonomy so every subspecialty appears in every split. Training uses augmented audio only (noise, speed, pitch variation) to help the model generalize, while val and test use clean originals so WER scores reflect real-world performance.

In [ ]:
from asr import package_dataset

splits = package_dataset(
    audio_manifest,
    text_dataset,
    train=0.8,
    val=0.1,
    test=0.1,
)
print(f"Train: {splits['train']}")
print(f"Val:   {splits['val']}")
print(f"Test:  {splits['test']}")

## Stage 4: Finetune

LoRA finetuning adapts Qwen3-ASR to your medical vocabulary by training small adapter layers while keeping the base model frozen. This runs locally on Apple Silicon via mlx-tune.

The per-device batch size is always 1 (audio models are too large for bigger batches on consumer hardware). `gradient_accumulation_steps` controls how many single-sample forward passes are summed before each weight update, simulating a larger effective batch.

Key parameters to adjust:
- `max_steps=20` — quick test. Remove for full training (steps are then computed from `epochs`). **When `max_steps` is set, `epochs` is ignored.**
- `epochs` — number of passes through the data (only used when `max_steps` is not set)
- `lr` — learning rate
- `lora_rank` — higher = more capacity, more overfitting risk
- `gradient_accumulation_steps` — effective batch size

The result is a lightweight adapter directory (~10 MB) that sits on top of the base model. To create a standalone model for uploading to HuggingFace, see the optional export cell at the end.

In [ ]:
from asr import finetune

model_path = finetune(
    splits["train"],
    splits["val"],
    max_steps=20,       # remove for full training
    # epochs=2,         # default: 2
    # lr=1e-4,          # default: 1e-4
    # lora_rank=8,      # default: 8
    # gradient_accumulation_steps=4,  # default: 4
)
print(f"Model saved to: {model_path}")

## Stage 5: Evaluate

Compares Word Error Rate (WER) between the base Qwen3-ASR and your finetuned version on the held-out test set. Lower is better — 0% means perfect transcription.

**Note:** With the quick-mode defaults (50 samples), the test set contains only a handful of samples. This is enough to verify the pipeline works end-to-end, but not enough for meaningful WER numbers. For reliable evaluation, generate enough samples so the test set is statistically meaningful.

In [ ]:
from asr import evaluate

results = evaluate(splits["test"], finetuned_model=str(model_path), language=language)
print(f"\nBase WER:      {results['base_wer']:.1%}")
if "finetuned_wer" in results:
    print(f"Finetuned WER: {results['finetuned_wer']:.1%}")
    improvement = results['base_wer'] - results['finetuned_wer']
    print(f"Improvement:   {improvement:.1%} absolute")

## Stage 6: Try It Yourself

Record a short audio clip of yourself dictating a medical finding, then transcribe it with your finetuned model.

In [ ]:
from pathlib import Path
from asr import transcribe

# Replace with your own audio file
recording = Path("my_recording.wav")
if recording.exists():
    text = transcribe(recording, model=str(model_path), language=language)
    print(text)
else:
    print(f"No recording found at '{recording}'. Record a WAV file and update the path above.")

## Optional: Export for HuggingFace

Merge the LoRA adapters into the base model to create a standalone checkpoint you can upload to HuggingFace or share with others.

In [ ]:
from asr import export_model

merged_path = export_model(model_path)
print(f"Merged model: {merged_path}")

# Upload to HuggingFace (uncomment and adjust)
# !huggingface-cli upload your-username/your-model-name {merged_path}

## What's Next?

Want to run your finetuned model on your phone for your clinical workflow? Visit [isaree.ai](https://isaree.ai) or reach out at hendrik [at] isaree [dot] ai.